- Ce script réalise une analyse approfondie de données d'enquête sur la pauvreté en Côte d'Ivoire (EHCVM 2021-2022), traitant à la fois des données socio-économiques et géographiques. Le processus commence par le chargement de deux fichiers Stata : un fichier principal contenant les informations socio-économiques des ménages (ehcvm_welfare_2b_CIV2021.dta) et un fichier avec leurs coordonnées GPS (s00_me_CIV2021.dta). Une exploration initiale est effectuée pour comprendre la structure des données, affichant les colonnes, dimensions et caractéristiques de chaque DataFrame.

- La fusion des données est réalisée sur trois variables clés : 'vague', 'grappe', et 'menage', permettant d'associer les coordonnées GPS à chaque ménage tout en préservant l'intégralité des données socio-économiques. Le script procède ensuite à un nettoyage méticuleux en identifiant, comptabilisant et supprimant les entrées avec des coordonnées GPS manquantes pour garantir la qualité des analyses géographiques ultérieures.

- Un élément central du script est la création de la variable binaire 'pcexp_binaire' qui catégorise les ménages comme pauvres (1) ou non pauvres (0) en comparant leurs dépenses par habitant (pcexp) au seuil de pauvreté national (zref) fixé à 369,516.46875 Francs CFA. Cette classification est fondamentale pour les analyses de pauvreté qui suivent.

- L'analyse exploratoire des données (EDA) est particulièrement exhaustive, utilisant diverses techniques de visualisation et d'analyse statistique : analyse des valeurs manquantes, calcul de statistiques descriptives, création de matrices de corrélation, visualisation des distributions par histogrammes, analyse des variables catégorielles, détection d'outliers via des box plots, et visualisation géographique des ménages sur une carte administrative de la Côte d'Ivoire avec geopandas.

- La conclusion du script est marquée par le calcul du taux de pauvreté national, utilisant une méthodologie sophistiquée qui prend en compte à la fois la taille des ménages (hhsize) et les pondérations de l'enquête (hhweight). Cette approche pondérée assure une estimation représentative et précise du taux de pauvreté au niveau national. Les résultats sont présentés sous forme de pourcentage et l'ensemble des données traitées est sauvegardé dans un fichier CSV pour permettre des analyses ultérieures.

In [1]:
# On charge les bibliothèques nécessaires
import pandas as pd
import numpy as np
import os

- La version finale des données de l'EHCVM sont données par le fichier "ehcvm_welfare_2b_CIV2021.dta" 

In [2]:
#Chargement des données d'enquetes
df_base =pd.read_stata(r'D:\wealth_predict_2021\data\DATA_EHCVM_2021_2022\Dataout\ehcvm_welfare_2b_CIV2021.dta')
df_gps =pd.read_stata(r'D:\wealth_predict_2021\data\DATA_EHCVM_2021_2022\Datain\Menage\s00_me_CIV2021.dta')

In [ ]:
for column in df_base:
    print(column)

In [ ]:
for column in df_gps:
    print(column)

In [5]:
print(df_base.shape)
print(df_gps.shape)

(12965, 36)
(13693, 76)


In [ ]:
df_base.head()

In [ ]:
df_gps.head()

In [ ]:
print(df_base.info())
print(df_gps.info())

In [9]:
# Fusion des deux DataFrames sur les clés communes 'vague', 'grappe' et 'menage'
merged_df = pd.merge(
    df_base,
    df_gps[['vague', 'grappe', 'menage', 'gps__latitude', 'gps__longitude']],
    on=['vague', 'grappe', 'menage'],
    how='left'  # Utilisation de 'left' pour garder toutes les lignes de df_base
)



In [ ]:

# Vérification du résultat
print(merged_df.head())
print(merged_df.shape)

In [ ]:
# Les informations sur les données fusionées
merged_df.info()

In [ ]:
# Identifier les lignes avec des valeurs manquantes pour 'gps__latitude' ou 'gps__longitude'
missing_gps = merged_df[merged_df['gps__latitude'].isnull() | merged_df['gps__longitude'].isnull()]
print("Lignes avec des valeurs GPS manquantes :")
print(missing_gps)

# Nombre de valeurs manquantes
print("Nombre total de valeurs manquantes :")
print(missing_gps.shape[0])


In [13]:
# Suppression des lignes avec des valeurs GPS manquantes
merged_df_cleaned = merged_df.dropna(subset=['gps__latitude', 'gps__longitude'])

# Vérifiez les dimensions après suppression
print("Dimensions après suppression :", merged_df_cleaned.shape)

Dimensions après suppression : (12952, 38)


- 'zref' est seuil de pauvreté et sa valeur est de 369 516.46875 Francs CFA

In [ ]:
print(merged_df_cleaned[['zref']].describe())

In [ ]:
# Création de la colonne pcexp_binaire
merged_df_cleaned['pcexp_binaire'] = (merged_df_cleaned['pcexp'] < merged_df_cleaned['zref']).astype(int)

# Vérification
print(merged_df_cleaned[['pcexp', 'zref', 'pcexp_binaire']].head())



In [16]:
# Définir le chemin et le nom du fichier CSV
output_path = r'D:\wealth_predict_2021\data\original_csv_file\Data_EHCVM_2021.csv'

# Sauvegarde du DataFrame au format CSV
merged_df_cleaned.to_csv(output_path, index=False, encoding='utf-8')

print(f"Le fichier a été sauvegardé avec succès à : {output_path}")


Le fichier a été sauvegardé avec succès à : D:\wealth_predict_2021\data\original_csv_file\Data_EHCVM_2021.csv


Cette portion du script identifie et corrige les doublons de coordonnées GPS dans un fichier de données en utilisant la méthode Haversine pour décaler légèrement les points dupliqués. Il commence par charger un fichier CSV (Data_EHCVM_2021.csv) contenant des données GPS et repère les lignes ayant des valeurs identiques de latitude et de longitude. Il regroupe ces doublons et affiche leurs coordonnées initiales ainsi que les index des observations concernées.
Ensuite, une fonction de décalage est appliquée pour ajuster les coordonnées des duplicatas en déplaçant les points dupliqués de 1 cm vers l'est. Pour chaque doublon, seules les occurrences répétées (à partir de la deuxième) sont ajustées. Les coordonnées initiales et ajustées sont affichées pour chaque point déplacé, permettant une vérification directe des changements.
Enfin, le script enregistre le DataFrame mis à jour dans un nouveau fichier CSV nommé Data_EHCVM_2021.csv, incluant les ajustements appliqués aux coordonnées GPS, pour éviter tout chevauchement dans les données.

In [21]:
!pip install haversine

In [22]:
import pandas as pd
import pandas as pd
from haversine import haversine, inverse_haversine, Direction

In [23]:
# Charger le fichier CSV
csv_path = r"D:\wealth_predict_2021\data\original_csv_file\Data_EHCVM_2021.csv"
df = pd.read_csv(csv_path)
# Identifier les doublons basés sur les colonnes 'GPS__Latitude' et 'GPS__Longitude'
duplicates = df[df.duplicated(subset=['gps__latitude', 'gps__longitude'], keep=False)]
# Afficher les coordonnées et les index des doublons
duplicates_info = duplicates.groupby(['gps__latitude', 'gps__longitude']).apply(lambda x: list(x.index))
for coords, indexes in duplicates_info.items():
    print(f"Coordonnées initiales des doublons: {coords} - Indexes: {indexes}")


Coordonnées initiales des doublons: (6.1423572, -5.9518134) - Indexes: [7221, 7228]


In [ ]:
# Fonction pour ajuster les coordonnées de 1 cm à l'est
def adjust_coordinates(latitude, longitude, distance_cm=1):
    distance_km = distance_cm / 100000  # Conversion de cm en km
    new_coords = inverse_haversine((latitude, longitude), distance_km, Direction.EAST)
    return new_coords

# Appliquer l'ajustement aux doublons (seulement à la deuxième occurrence et plus)
for coords, indexes in duplicates_info.items():
    for idx in indexes[1:]:  # Évite la première occurrence
        lat, lon = df.loc[idx, 'gps__latitude'], df.loc[idx, 'gps__longitude']
        
        # Afficher les coordonnées avant le décalage
        print(f"Avant décalage - Index: {idx}, latitude: {lat}, longitude: {lon}")
        
        # Calculer les nouvelles coordonnées
        new_lat, new_lon = adjust_coordinates(lat, lon)
        
        # Appliquer le décalage au DataFrame
        df.at[idx, 'gps__latitude'] = new_lat
        df.at[idx, 'gps__longitude'] = new_lon
        
        # Afficher les coordonnées après le décalage
        print(f"Après décalage - Index: {idx}, Nouvelle latitude: {new_lat}, Nouvelle longitude: {new_lon}")

# Afficher les coordonnées mises à jour pour vérifier les changements
print("Coordonnées mises à jour :")
print(df[['gps__latitude', 'gps__longitude']])


In [25]:
# Sauvegarder le DataFrame en tant que CSV
output_path = r"D:\wealth_predict_2021\data\original_csv_file\Data_EHCVM_2021.csv"
df.to_csv(output_path, index=False)

##### Exploratory data analysis

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point

In [18]:
# Chargement des données nettoyées (si nécessaire)

# merged_df_cleaned = pd.read_csv("path_to_merged_df_cleaned.csv")


In [ ]:
# Mettre à jour les types des variables spécifiques
merged_df_cleaned['grappe'] = merged_df_cleaned['grappe'].astype('category')
merged_df_cleaned['vague'] = merged_df_cleaned['vague'].astype('category')
merged_df_cleaned['year'] = merged_df_cleaned['year'].astype('category')
merged_df_cleaned['hhid'] = merged_df_cleaned['hhid'].astype(str)  # hhid est un numéro d'identification du ménage

# Aperçu des données
print("Aperçu des premières lignes :")
print(merged_df_cleaned.head())

# 1. Analyse des valeurs manquantes
missing_values = merged_df_cleaned.isnull().mean() * 100
print("\nPourcentage de valeurs manquantes :")
print(missing_values[missing_values > 0].sort_values(ascending=False))

# 2. Statistiques descriptives pour les variables continues
print("\nRésumé statistique des variables continues :")
continuous_columns = merged_df_cleaned.select_dtypes(include=['float64', 'float32']).columns
continuous_columns = [col for col in continuous_columns if col not in ['gps__longitude', 'gps__latitude', 'zref', 'def_temp', 'def_spa']]
print(merged_df_cleaned[continuous_columns].describe())

# 3. Corrélations entre les variables continues
print("\nMatrice de corrélation :")
correlation_matrix = merged_df_cleaned[continuous_columns].corr()
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Matrice de corrélation des variables continues (sans Zref, GPS, def_temp, def_spa)")
plt.show()

# 4. Distribution des variables continues
for col in continuous_columns:
    plt.figure(figsize=(12, 6))
    sns.histplot(merged_df_cleaned[col], kde=True, bins=30)
    plt.title(f"Distribution de {col}")
    plt.xlabel(col)
    plt.ylabel("Fréquence")
    plt.show()

# 5. Répartition des variables catégoriques
categorical_columns = [col for col in merged_df_cleaned.select_dtypes(include=['category', 'object']).columns if col not in ['country', 'year', 'hhid', 'grappe']]
for col in categorical_columns:
    plt.figure(figsize=(10, 8))
    sns.countplot(data=merged_df_cleaned, x=col, order=merged_df_cleaned[col].value_counts().index)
    plt.title(f"Répartition de {col}")
    plt.xticks(rotation=90)
    plt.show()

# 6. Relation entre variables catégoriques et continues
if 'region' in merged_df_cleaned.columns and 'pcexp' in merged_df_cleaned.columns:
    plt.figure(figsize=(12, 8))
    sns.boxplot(data=merged_df_cleaned, x='region', y='pcexp')
    plt.title("Répartition de pcexp par région")
    plt.xticks(rotation=90)
    plt.show()

# 7. Détection des outliers
for col in continuous_columns:
    plt.figure(figsize=(12, 8))
    sns.boxplot(data=merged_df_cleaned, x=col)
    plt.title(f"Détection des outliers dans {col}")
    plt.xlabel(col)
    plt.show()

# 8. Visualisation géographique des points GPS
if 'gps__latitude' in merged_df_cleaned.columns and 'gps__longitude' in merged_df_cleaned.columns:
    # Créer une GeoDataFrame avec les coordonnées GPS
    geometry = [Point(xy) for xy in zip(merged_df_cleaned['gps__longitude'], merged_df_cleaned['gps__latitude'])]
    geo_df = gpd.GeoDataFrame(merged_df_cleaned, geometry=geometry, crs="EPSG:4326")  # Système WGS84

    # Charger les découpages administratifs de la Côte d'Ivoire
    gadm_path = r'D:\wealth_predict_2021\data\downloaded\gadm41_CIV_4.json'
    civ = gpd.read_file(gadm_path)

    # Tracer les points sur la carte des découpages administratifs
    fig, ax = plt.subplots(figsize=(12, 12))
    civ.plot(ax=ax, color='lightgrey', edgecolor='black', alpha=0.7)
    geo_df.plot(ax=ax, markersize=10, color='red', alpha=0.9)
    plt.title("Points GPS sur les découpages administratifs de la Côte d'Ivoire")
    plt.show()


##### calcul du taux de pauvreté

In [20]:
import pandas as pd

# Supposons que `merged_df_cleaned` est votre DataFrame contenant les colonnes nécessaires

# Calcul intermédiaire pour chaque ménage
merged_df_cleaned['weighted_poverty'] = merged_df_cleaned['hhsize'] * merged_df_cleaned['hhweight'] * merged_df_cleaned['pcexp_binaire']
merged_df_cleaned['total_weight'] = merged_df_cleaned['hhsize'] * merged_df_cleaned['hhweight']

# Calcul du taux de pauvreté réel
total_weighted_poverty = merged_df_cleaned['weighted_poverty'].sum()
total_weight = merged_df_cleaned['total_weight'].sum()
poverty_rate = total_weighted_poverty / total_weight

# Affichage du résultat
print(f"Taux de pauvreté réel : {poverty_rate:.2%}")
print(f"Taux de pauvreté réel arrondi : {poverty_rate:.1%}")



Taux de pauvreté réel : 37.46%
Taux de pauvreté réel arrondi : 37.5%


C:\Users\k.kone\AppData\Local\Temp\ipykernel_22732\2780417484.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df_cleaned['weighted_poverty'] = merged_df_cleaned['hhsize'] * merged_df_cleaned['hhweight'] * merged_df_cleaned['pcexp_binaire']
C:\Users\k.kone\AppData\Local\Temp\ipykernel_22732\2780417484.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df_cleaned['total_weight'] = merged_df_cleaned['hhsize'] * merged_df_cleaned['hhweight']
